# Clase 139 — Generación de texto char-RNN

Construimos un **modelo de lenguaje autoregresivo a nivel carácter** (Karpathy 2015):
**next-token prediction**, la misma tarea de pre-training de todo LLM moderno. Vemos
**sampling** con **temperatura** y **top-k** para controlar la creatividad.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Corpus y vocabulario de caracteres

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
texto = ("ROMEO: But soft, what light through yonder window breaks? "
         "It is the east, and Juliet is the sun. "
         "Arise fair sun and kill the envious moon. ") * 40
vocab = sorted(set(texto))
stoi = {c: i for i, c in enumerate(vocab)}      # char -> int
itos = {i: c for c, i in stoi.items()}          # int -> char
vocab_size = len(vocab)
print("longitud corpus:", len(texto), "| vocab_size:", vocab_size)

## 2. Ventanas `(entrada, target)` con el carácter siguiente

In [ ]:
datos = np.array([stoi[c] for c in texto], dtype="int32")
T = 40
X = np.stack([datos[i:i + T] for i in range(len(datos) - T - 1)])
y = np.stack([datos[i + 1:i + T + 1] for i in range(len(datos) - T - 1)])  # target = shift +1
print("X:", X.shape, "| y:", y.shape)   # se predice el siguiente char en cada posición

## 3. Modelo `Embedding → GRU → Dense(vocab)`

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(T,)),
    layers.Embedding(vocab_size, 64),
    layers.GRU(128, return_sequences=True),   # salida por cada timestep
    layers.Dense(vocab_size),                 # logits sobre el vocabulario
])
modelo.summary()

## 4. Entrenamiento con cross-entropy sobre logits

In [ ]:
modelo.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
modelo.fit(X, y, epochs=5, batch_size=64, verbose=2)

## 5. Sampling autoregresivo con temperatura

In [ ]:
def generar(modelo, semilla, n=200, temp=1.0):
    ids = [stoi[c] for c in semilla]
    gen = np.random.default_rng(0)
    for _ in range(n):
        ventana = np.array(ids[-T:])[None]
        logits = modelo.predict(ventana, verbose=0)[0, -1]   # logits del último paso
        logits = logits / temp                               # temperatura: T<1 confiado, T>1 diverso
        p = np.exp(logits - logits.max())
        p /= p.sum()
        ids.append(int(gen.choice(len(p), p=p)))
    return "".join(itos[i] for i in ids)

print(generar(modelo, "ROMEO:", n=120, temp=0.5)[:120])

## 6. Top-k sampling

In [ ]:
def muestrear_top_k(logits, k=5, temp=1.0):
    logits = logits / temp
    idx = np.argpartition(logits, -k)[-k:]     # los k más probables
    sub = logits[idx]
    p = np.exp(sub - sub.max())
    p /= p.sum()
    return int(np.random.default_rng(0).choice(idx, p=p))

demo = np.random.default_rng(1).standard_normal(vocab_size)
print("token elegido con top-k=5:", repr(itos[muestrear_top_k(demo, k=5)]))

## Ejercicios

1. **Vocab + encoding**: tokenizá el corpus a enteros y reportá `vocab_size`.
2. **Sampling con temperatura**: generá 200 chars a `temp ∈ {0.3, 0.7, 1.2}` y compará.
3. **Top-k**: implementá `top-k` con `np.argpartition` y observá cómo reduce el gibberish.
4. **Greedy vs sampling**: reemplazá el muestreo por `argmax` y verificá los loops repetitivos.

## Conclusiones

- **Next-token prediction** es self-supervised: el mismo objetivo que GPT, a otra escala.
- El vocabulario char evita OOV pero produce secuencias largas; en producción se usa **BPE**.
- La **temperatura** divide los logits: `T<1` más determinista, `T>1` más diverso.
- **Top-k** (y top-p/nucleus) recorta la cola de baja probabilidad y mejora la calidad.
- `argmax` greedy cae en loops ("the the the"); el sampling aporta diversidad.